# 02 – Preprocessing
**CS 5331/7331 · Project Team Five**

1. Load the dataset and check data quality (missing values, duplicates)
2. Encode the label (phishing = 1, legitimate = 0)
3. Define feature sets: full (30) and no-external-lookups (25)
4. Stratified 80/20 train/test split
5. SMOTEN oversampling on the training set only
6. Scaling decision (applied inside model pipelines)
7. Save all splits to /data for the modeling notebooks

In [16]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imbalanced-learn

import json
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTEN

BASE = '/content/drive/MyDrive/CS7331_Phishing'
DATA_DIR = f'{BASE}/data'
SEED = 42  # same seed everywhere so we get the same results every run

df = pd.read_csv(f'{DATA_DIR}/phishing.csv')
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(11055, 31)


In [17]:
# quick check for anything missing or repeated
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate rows (features only):", df.drop(columns='result').duplicated().sum())

Missing values: 0
Duplicate rows: 5206
Duplicate rows (features only): 5270


In [18]:
# almost half the rows are exact copies, which would leak into the test set and inflate scores
df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

# new class balance, since dropping rows can shift it
print(df['result'].map({-1: 'Phishing', 1: 'Legitimate'}).value_counts(normalize=True).round(3))

# rows with identical features but different labels (we keep these, they're real ambiguity)
conflicts = df.drop(columns='result').duplicated(keep=False).sum()
print("Rows in conflicting groups:", conflicts)

After removing duplicates: (5849, 31)
result
Phishing      0.516
Legitimate    0.484
Name: proportion, dtype: float64
Rows in conflicting groups: 128


In [19]:
# flip the label so phishing = 1, since that's the class we're trying to catch
y = (df['result'] == -1).astype(int).rename('phishing')
X = df.drop(columns='result')

# these 5 need outside services (WHOIS, Google, etc.), so the second set drops them
EXTERNAL = ['age_of_domain', 'dnsrecord', 'web_traffic', 'page_rank', 'google_index']
FEATURE_SETS = {
    'full': list(X.columns),
    'no_external': [c for c in X.columns if c not in EXTERNAL],
}
for name, cols in FEATURE_SETS.items():
    print(f"{name}: {len(cols)} features")

# save the column lists so the model notebooks can grab either set
with open(f'{DATA_DIR}/feature_sets.json', 'w') as f:
    json.dump(FEATURE_SETS, f, indent=2)

full: 30 features
no_external: 25 features


In [20]:
# 80/20 split, stratified so both sides keep the same phishing ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Phishing share - full:", round(y.mean(), 3),
      " train:", round(y_train.mean(), 3),
      " test:", round(y_test.mean(), 3))

Train: (4679, 30)  Test: (1170, 30)
Phishing share - full: 0.516  train: 0.516  test: 0.516


In [21]:
# dropping the 5 external features can make rows identical again, check how many land in both train and test
ne = FEATURE_SETS['no_external']
print("Duplicates within 25 features:", X[ne].duplicated().sum())
overlap = X_test[ne].merge(X_train[ne].drop_duplicates(), how='inner')
print("Test rows with an exact copy in train (25 features):", len(overlap))

Duplicates within 25 features: 2280
Test rows with an exact copy in train (25 features): 597


In [26]:
# how many of those look-alikes have opposite labels? the 25-feature models can't get these right
conf_ne = pd.concat([X[ne], y], axis=1).drop_duplicates()
conf_ne = conf_ne[ne].duplicated(keep=False).sum()
print("Rows in conflicting groups (25 features):", conf_ne, "vs 128 with all 30")

Rows in conflicting groups (25 features): 184 vs 128 with all 30


In [22]:
# balance the classes in the training set only, the test set stays untouched
# SMOTEN instead of regular SMOTE since our features are categories, not numbers
smote = SMOTEN(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_sm.value_counts().to_dict())

# make sure the fake rows only use real values (-1, 0, 1), nothing weird like 0.37
bad = [c for c in X_train_sm.columns
       if not set(X_train_sm[c].unique()) <= set(X[c].unique())]
print("Columns with invalid values:", bad if bad else "none")

Before SMOTE: {1: 2415, 0: 2264}
After SMOTE:  {1: 2415, 0: 2415}
Columns with invalid values: none


In [23]:
# separate SMOTE run for the 25-feature set so it only uses those features
X_train_sm_ne, y_train_sm_ne = SMOTEN(random_state=SEED).fit_resample(X_train[ne], y_train)
print("No-external after SMOTE:", y_train_sm_ne.value_counts().to_dict())

No-external after SMOTE: {1: 2415, 0: 2415}


In [24]:
# everything's already between -1 and 1, so scaling won't change much
# we'll scale inside the kNN/SVM pipelines instead of here
print(X_train.describe().loc[['mean', 'std', 'min', 'max']].T.round(2).head(10))

                            mean   std  min  max
having_ip_address           0.15  0.99 -1.0  1.0
url_length                 -0.62  0.78 -1.0  1.0
shortining_service          0.72  0.69 -1.0  1.0
having_at_symbol            0.59  0.80 -1.0  1.0
double_slash_redirecting    0.72  0.70 -1.0  1.0
prefix_suffix              -0.76  0.65 -1.0  1.0
having_sub_domain           0.06  0.81 -1.0  1.0
sslfinal_state              0.14  0.93 -1.0  1.0
domain_registration_length -0.30  0.95 -1.0  1.0
favicon                     0.59  0.81 -1.0  1.0


In [27]:
# save all the splits so Daniel's model notebooks use the exact same data
X_train.to_csv(f'{DATA_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{DATA_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{DATA_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{DATA_DIR}/y_test.csv', index=False)
X_train_sm.to_csv(f'{DATA_DIR}/X_train_smote.csv', index=False)
y_train_sm.to_csv(f'{DATA_DIR}/y_train_smote.csv', index=False)
X_train_sm_ne.to_csv(f'{DATA_DIR}/X_train_smote_noext.csv', index=False)
y_train_sm_ne.to_csv(f'{DATA_DIR}/y_train_smote_noext.csv', index=False)
print("Saved 8 split files + feature_sets.json to /data")


Saved 8 split files + feature_sets.json to /data
